# Milestone 4 – Benchmark and Recommendation

| Required evidence | Section |
|---|---|
| Shared held-out benchmark | 1–2 |
| Aggregate and per-case comparison | 3–4 |
| Failure analysis | 5 |
| A recommendation supported by measured results | 6 |

The benchmark is in `src/evaluation.py`. It writes `reports/evaluation_results.json` (the full trace) and `reports/evaluation_report.md`.

```bash
uv run python -m src.evaluation               # run the benchmark (a few minutes on CPU)
uv run python -m src.evaluation --report-only # rebuild the report from the saved results
```

In [ ]:
import json
import os
from pathlib import Path

# Run from the project root so data/, src/, artifacts/ and reports/ resolve.
if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.datasets import load_evaluation
from src.evaluation import REPORT_PATH, RESULTS_PATH, build_report, run_benchmark

print("Working directory:", Path.cwd())

## 1. Run or load the benchmark

Set `RERUN = True` to answer all 14 held-out cases with both systems again. Otherwise the saved results are loaded. Greedy decoding makes reruns deterministic apart from latency.

In [ ]:
RERUN = False

if RERUN or not RESULTS_PATH.exists():
    results = run_benchmark()
    RESULTS_PATH.parent.mkdir(exist_ok=True)
    RESULTS_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")
    REPORT_PATH.write_text(build_report(results), encoding="utf-8")
else:
    results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))

systems = results["systems"]
names = list(systems)
cases = {c["id"]: c for c in results["cases"]}
supported = [c for c in cases.values() if not c["should_abstain"]]
print(f"{len(cases)} held-out cases: {len(supported)} supported, {len(cases) - len(supported)} unsupported")
print("Systems:", names)

In [ ]:
# Both systems must use the same configuration except for the adapter.
for section, values in results["config"].items():
    print(f"[{section}]")
    for key, value in values.items():
        if key not in ("epoch_losses", "system_instruction"):
            print(f"  {key:<22} {value}")

## 2. Retrieval (evaluated once, shared by both systems)

In [ ]:
retrieval = results["retrieval"]
print(f"recall@{retrieval['top_k']}: {retrieval['recall_at_k']:.3f}   MRR: {retrieval['mrr']:.3f}   "
      f"retrieval-only decision accuracy: {retrieval['decision_accuracy']:.3f}")
print()
print(f"{'case':<28} {'abstain?':>8} {'top':>6}  retrieved")
for row in retrieval["per_case"]:
    print(f"{row['id']:<28} {str(row['should_abstain']):>8} {row['top_score']:>6.3f}  {row['retrieved']}")

## 3. Aggregate comparison

The rubric is defined in `src/evaluation.py` and in section 2 of the report. *Raw coverage* is measured **before** citation checks, so it separates content from citation format.

In [ ]:
metrics = [
    ("answered", "{:.0f}"), ("abstained", "{:.0f}"), ("rejected", "{:.0f}"),
    ("relevance", "{:.3f}"), ("raw_coverage_supported", "{:.3f}"),
    ("faithfulness", "{:.3f}"), ("style", "{:.3f}"), ("decision_accuracy", "{:.3f}"),
    ("mean_latency", "{:.2f}"), ("median_generation_latency", "{:.2f}"),
]
base, hybrid = (systems[n]["aggregate"] for n in names)
print(f"{'metric':<27} {names[0]:>14} {names[1]:>16} {'delta':>8}")
for key, fmt in metrics:
    print(f"{key:<27} {fmt.format(base[key]):>14} {fmt.format(hybrid[key]):>16} "
          f"{hybrid[key] - base[key]:>+8.3f}")

## 4. Per-case comparison

In [ ]:
rows = {name: {r["id"]: r for r in systems[name]["cases"]} for name in names}
print(f"{'case':<28} {'system':<16} {'status':<10} {'rel':>5} {'faith':>5} {'style':>5} {'ok':>3}  citations")
for case_id in cases:
    for name in names:
        a, s = rows[name][case_id]["answer"], rows[name][case_id]["score"]
        print(f"{case_id if name == names[0] else '':<28} {name:<16} {a['status']:<10} "
              f"{s['relevance']:>5.2f} {'yes' if s['faithful'] else 'no':>5} {s['style']:>5.2f} "
              f"{'yes' if s['decision_correct'] else 'no':>3}  {', '.join(a['citations']) or '-'}")

In [ ]:
# Content vs format: baseline answers that had the right terms but were rejected for missing citations.
print("Baseline raw answers covering all required terms, but rejected:")
for case in supported:
    r = rows[names[0]][case["id"]]
    if r["score"]["raw_coverage"] == 1.0 and r["answer"]["status"] == "rejected":
        print(f"  {case['id']:<26} {r['answer']['raw'][:90]}")

## 5. Failure analysis

Reading the raw answers matters because the metrics are lexical. These four cases show where the numbers mislead.

In [ ]:
def show_case(case_id, note):
    case = cases[case_id]
    print(f"=== {case_id}: {case['question']}")
    print(f"    expected: {case['expected_sources'] or 'ABSTAIN'}  required: {case['required_terms']}")
    for name in names:
        r = rows[name][case_id]
        print(f"  [{name}] {r['answer']['status']} | faithful={r['score']['faithful']}")
        print(f"     {r['answer']['raw'] or '(not generated)'}")
    print(f"  -> {note}\n")

show_case("eval-bank-chargeback",
          "Unsupported question passed the threshold; the hybrid invented a claim with a valid citation and it was SERVED.")
show_case("eval-packed-address",
          "Hybrid answer is evasive ('before' packing) but counts as faithful: the proxy only checks citations and exact forbidden phrases.")
show_case("eval-log-secrets",
          "Both say logs MAY contain tokens (policy: MUST NOT); the forbidden phrase differs in wording, so it is not detected.")
show_case("eval-authenticator",
          "Hybrid answer misses both required facts and cites unrelated sources, yet counts as faithful.")

## 6. Recommendation

The full analysis (failures, retrieval limits, cost, recommendation, and answers to the project questions) is in `reports/analysis.md` and is shown below.

In [ ]:
from IPython.display import Markdown

Markdown(Path("reports/analysis.md").read_text(encoding="utf-8"))